# EfficientNet with Augmentation Aware Training

Using a pretrained EfficientNet-B0 model, trained on augmented pictures from the dataset, to match the problem statement's 6 transform conditions (JPEG, blur, downscale, noise, color jitter, center crop).

In [1]:
# Installing timm for the pretrained EffNet backbone, albumentations image augmentation pipeline
!pip install -q timm albumentations

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from tqdm.notebook import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, confusion_matrix,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, brier_score_loss,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Dataset paths in Drive
TRAIN_REAL_DIR = "/content/drive/MyDrive/sidset_subset/train/REAL"
TRAIN_FAKE_DIR = "/content/drive/MyDrive/sidset_subset/train/FAKE"
VAL_REAL_DIR   = "/content/drive/MyDrive/sidset_subset/test/REAL"
VAL_FAKE_DIR   = "/content/drive/MyDrive/sidset_subset/test/FAKE"
CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

IMG_SIZE = 224
CANONICAL_SIZE = 256     # images resized here first, then cropped/augmented, then back down to IMG_SIZE
BATCH_SIZE = 32
NUM_EPOCHS = 5
LR = 1e-4                # kept small, not training from scratch, using frozen backbone
DEBUG_SUBSET_SIZE = 200  # small subset for a sanity run

Device: cuda


In [16]:
# Mounting my dataset folder into Colab files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Augmentation: Matches the problem statement's 6 augmentation modes.
# One augmentation applied per image. Approx 40% kept clean.

# 1. Resizing before crop
crop_size = int(CANONICAL_SIZE * 0.8)  # 80% crop

# 2. Augmenting training data
train_transform = A.Compose([
    A.Resize(CANONICAL_SIZE, CANONICAL_SIZE),
    A.OneOf([
        A.ImageCompression(quality_range=(25, 95), p=1.0),                      # JPEG compression
        A.GaussianBlur(blur_limit=(3, 9), sigma_limit=(0.4, 2.1), p=1.0),       # Gaussian Blur
        A.Downscale(scale_range=(0.2, 0.55), p=1.0),                            # Downscale and upscale back
        A.GaussNoise(std_range=(0.02, 0.12), p=1.0),                            # Gaussian Noise
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0, p=1.0), # Color jitter
        A.CenterCrop(height=crop_size, width=crop_size, p=1.0),                 # Center crop
    ], p=0.6),  # 40% of samples pass through with no augmentation
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),         # ImageNet stats, since EffNet backbone was trained on ImageNet
    ToTensorV2(),
])

# Test/validation data kept clean. Robustness check will be conducted separately
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),         # ImageNet stats
    ToTensorV2(),
])

In [5]:
# Dataset: standard Pytorch

class RealFakeDataset(Dataset):
    """Loads real/fake images from two directories. Paths are resolved once
    in __init__ and indexed directly by idx in __getitem__."""

    def __init__(self, real_dir, fake_dir, transform=None):         # Used to initiate list of file paths, adds label to each image
        self.transform = transform
        real_paths = [os.path.join(real_dir, f) for f in sorted(os.listdir(real_dir))
                      if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        fake_paths = [os.path.join(fake_dir, f) for f in sorted(os.listdir(fake_dir))
                      if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        self.samples = [(p, 0) for p in real_paths] + [(p, 1) for p in fake_paths]  # 0=real, 1=fake

    def __len__(self):          # To count how many samples are inside
        return len(self.samples)

    def __getitem__(self, idx):         # Image loading, converts to RGB, then runs through augmentation pipeline
        path, label = self.samples[idx]
        image = np.array(Image.open(path).convert("RGB"))
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return image, torch.tensor(label, dtype=torch.float32), path

In [6]:
train_dataset = RealFakeDataset(TRAIN_REAL_DIR, TRAIN_FAKE_DIR, transform=train_transform)    # Creating 2 instances of RealFakeDataset for training and validation
val_dataset   = RealFakeDataset(VAL_REAL_DIR, VAL_FAKE_DIR, transform=val_transform)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

# Small subset sanity check
debug_indices = random.sample(range(len(train_dataset)), min(DEBUG_SUBSET_SIZE, len(train_dataset)))
debug_loader = DataLoader(Subset(train_dataset, debug_indices), batch_size=8, shuffle=True)

imgs, labels, paths = next(iter(debug_loader))
# Checking for Shape, Pixel range, Label
print("Batch image shape:", imgs.shape, "| dtype:", imgs.dtype)
print("Pixel value range:", imgs.min().item(), "to", imgs.max().item())
print("Label distribution in batch:", labels.tolist())

# Actual loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

Train samples: 16000 | Val samples: 2647
Batch image shape: torch.Size([8, 3, 224, 224]) | dtype: torch.float32
Pixel value range: -2.1179039478302 to 2.6399998664855957
Label distribution in batch: [1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0]


In [7]:
# Building model

def build_efficientnet(freeze_until_last_n_blocks=1):
    """
    Loads ImageNet-pretrained EfficientNet-B0, freezes everything except the last
    'freeze_until_last_n_blocks' MBConv stages plus the head
    conv and classifier. Training adapts high-level features to
    real/fake artifacts instead of relearning low-level filters from scratch.
    """
    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1)

    for param in model.parameters():
        param.requires_grad = False           # False freezes every param in layers of the model

    for stage in model.blocks[-freeze_until_last_n_blocks:]:
        for param in stage.parameters():
            param.requires_grad = True        # Unfreezes last n block params

    for module in [model.conv_head, model.bn2, model.classifier]:
        for param in module.parameters():
            param.requires_grad = True        # Unfreezes conv head, bn2, classifier (outputs logits)

    return model

model = build_efficientnet(freeze_until_last_n_blocks=1).to(DEVICE)     # trains classifier, using GPU

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")    # Confirm freezing by number of trainable params

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Trainable params: 1,130,673 / 4,008,829 (28.2%)


In [8]:
# Loss function, takes in logits, uses binary cross entropy, applies sigmoid
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(                                  # Updates weights
    filter(lambda p: p.requires_grad, model.parameters()),      # Only filters out unfrozen layers to be updated
    lr=LR, weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [9]:
# TRAINING, 5 epochs

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_probs, all_labels, all_paths = [], [], []         # Initialising empty accumulators

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels, paths in tqdm(loader, desc='Training' if train else 'Validation'):          # tqdm wrapping to run progress bars
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs).squeeze(1)

            # learning
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            all_paths.extend(paths)         # converting raw logits into probability

    avg_loss = total_loss / len(loader.dataset)
    auc = roc_auc_score(all_labels, all_probs)

    prob_std = float(np.std(all_probs))
    if prob_std < 1e-3:
        print(f"  WARNING: predicted probabilities have almost no spread (std={prob_std:.6f}) "         # sanity check
              f" Check for a data pipeline bug before trusting this AUC.")

    return avg_loss, auc, all_probs, all_labels, all_paths

In [10]:
'''
Training loop. Intialises best AUC value for tracking, then loops through number of epochs (5), with train = True.
run_epoch returns 5 things: avg_loss, auc, all_probs, all_labels, all_paths, but we only want loss and AUC for this step, rest are omitted.
scheduler.step() updates the learning rate for the next epoch.
Includes steps for progress visualizers and checkpointing, before finally printing the best AUC achieved.
'''
best_val_auc = 0.0

print("Starting training...")
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_auc, _, _, _ = run_epoch(model, train_loader, train=True)
    val_loss, val_auc, val_probs, val_labels, val_paths = run_epoch(model, val_loader, train=False)
    scheduler.step()

    print(f"Epoch {epoch}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_auc={train_auc:.4f} | "
          f"val_loss={val_loss:.4f} val_auc={val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "efficientnet_best.pt"))      # model.state_dict() is a dict of the current weight tensors
        print(f"  -> new best model saved (val_auc={val_auc:.4f})")

print(f"\nBest val AUC: {best_val_auc:.4f}")

Starting training...


Training:   0%|          | 0/500 [00:00<?, ?it/s]

Validation:   0%|          | 0/83 [00:00<?, ?it/s]

Epoch 1/5 | train_loss=0.5503 train_auc=0.9434 | val_loss=0.2273 val_auc=0.9832
  -> new best model saved (val_auc=0.9832)


Training:   0%|          | 0/500 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/83 [00:00<?, ?it/s]

Epoch 4/5 | train_loss=0.1082 train_auc=0.9944 | val_loss=0.1260 val_auc=0.9933
  -> new best model saved (val_auc=0.9933)


Training:   0%|          | 0/500 [00:00<?, ?it/s]

Validation:   0%|          | 0/83 [00:00<?, ?it/s]

Epoch 5/5 | train_loss=0.0814 train_auc=0.9963 | val_loss=0.1162 val_auc=0.9940
  -> new best model saved (val_auc=0.9940)

Best val AUC: 0.9940


## Export raw probabilities for the fusion pipeline

Reloads the best checkpoint and exports unthresholded probabilities on
the validation set. Feed this CSV into the fusion pipeline logistic regression
with CLIP and DCT raw probabilities.

In [11]:
'''
Saving models weights during training, by the final epoch. model.load_state_dict saves updated weights into model.
train=False to stop weights from changing
results_df builds a table (pandas dataframe) with one row per validation image
3 columns: image path, true label (0=real, 1=fake), the predicted probability of it being fake
Generates .csv output for fusion pipeline
'''

model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "efficientnet_best.pt")))
_, final_auc, final_probs, final_labels, final_paths = run_epoch(model, val_loader, train=False)

results_df = pd.DataFrame({
    "image_path": final_paths,
    "label": final_labels,
    "efficientnet_prob": final_probs,
})
results_df.to_csv(os.path.join(CHECKPOINT_DIR, "efficientnet_val_probs.csv"), index=False)
print(f"Saved {len(results_df)} raw probabilities for the fusion pipeline. Final val AUC: {final_auc:.4f}")

Validation:   0%|          | 0/83 [00:00<?, ?it/s]

Saved 2647 raw probabilities for the fusion pipeline. Final val AUC: 0.9940


In [12]:
'''
More in depth validation metrics beyond AUC. Reload best weights, get raw probs (0=real, 1=fake)
map_location=DEVICE explicitly tells torch.load which device to put the tensors on when loading
'''

model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "efficientnet_best.pt"),
                                 map_location=DEVICE))
_, _, probs, labels, _ = run_epoch(model, val_loader, train=False)
probs  = np.asarray(probs, dtype=float)
labels = np.asarray(labels, dtype=int)

def pick_threshold_at_fpr(labels, probs, target_fpr=0.05):
    """
    Highest-precision threshold whose real-image FPR <= target.
    For the real eval, pick this on a held-out clean split and freeze it
    across every augmentation type
    """
    fpr, tpr, thr = roc_curve(labels, probs)
    ok = np.where(fpr <= target_fpr)[0]
    return float(thr[ok[-1]]) if len(ok) else 0.5

def report_metrics(labels, probs, threshold, title=""):
    '''
    Confusion matrix and 10 other validation metrics
    '''
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    rows = {
        "ROC-AUC (ranking, threshold-free)": roc_auc_score(labels, probs),
        "PR-AUC / Average Precision":        average_precision_score(labels, probs),
        "Accuracy":                          accuracy_score(labels, preds),
        "Balanced accuracy":                 balanced_accuracy_score(labels, preds),
        "Precision (of fake calls)":         precision_score(labels, preds, zero_division=0),
        "Recall / TPR (fakes caught)":       recall_score(labels, preds, zero_division=0),
        "Specificity / TNR (reals kept)":    tn / (tn + fp) if (tn + fp) else float("nan"),
        "FPR (REAL flagged as fake)":        fp / (tn + fp) if (tn + fp) else float("nan"),
        "F1 (fake class)":                   f1_score(labels, preds, zero_division=0),
        "Brier score (calibration, lower=better)": brier_score_loss(labels, probs),
    }
    print(f"\n=== {title} | threshold = {threshold:.3f} ===")
    for k, v in rows.items():
        print(f"  {k:42s}: {v:.4f}")
    print(f"  confusion [TN FP / FN TP]                 : [{tn} {fp} / {fn} {tp}]")
    return rows

# Running full report twice: Once with 0.5 threshold and one with calibrated threshold
thr = pick_threshold_at_fpr(labels, probs, target_fpr=0.05)
report_metrics(labels, probs, threshold=0.5, title="Default 0.5")
report_metrics(labels, probs, threshold=thr, title="Calibrated @ 5% target FPR")

Validation:   0%|          | 0/83 [00:00<?, ?it/s]


=== Default 0.5 | threshold = 0.500 ===
  ROC-AUC (ranking, threshold-free)         : 0.9940
  PR-AUC / Average Precision                : 0.9941
  Accuracy                                  : 0.9660
  Balanced accuracy                         : 0.9660
  Precision (of fake calls)                 : 0.9639
  Recall / TPR (fakes caught)               : 0.9683
  Specificity / TNR (reals kept)            : 0.9637
  FPR (REAL flagged as fake)                : 0.0363
  F1 (fake class)                           : 0.9661
  Brier score (calibration, lower=better)   : 0.0272
  confusion [TN FP / FN TP]                 : [1276 48 / 42 1281]

=== Calibrated @ 5% target FPR | threshold = 0.292 ===
  ROC-AUC (ranking, threshold-free)         : 0.9940
  PR-AUC / Average Precision                : 0.9941
  Accuracy                                  : 0.9645
  Balanced accuracy                         : 0.9645
  Precision (of fake calls)                 : 0.9515
  Recall / TPR (fakes caught)             

{'ROC-AUC (ranking, threshold-free)': np.float64(0.9940113675547426),
 'PR-AUC / Average Precision': np.float64(0.9941378986446882),
 'Accuracy': 0.9644880997355497,
 'Balanced accuracy': np.float64(0.9644935181188958),
 'Precision (of fake calls)': 0.9515062454077884,
 'Recall / TPR (fakes caught)': 0.9788359788359788,
 'Specificity / TNR (reals kept)': np.float64(0.9501510574018127),
 'FPR (REAL flagged as fake)': np.float64(0.04984894259818731),
 'F1 (fake class)': 0.9649776453055141,
 'Brier score (calibration, lower=better)': np.float64(0.027150159289695965)}

In [19]:
import shutil

# Export the trained model in a few portable formats and verify a clean reload
BEST_CKPT = os.path.join(CHECKPOINT_DIR, "efficientnet_best.pt")
export_model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=1)
export_model.load_state_dict(torch.load(BEST_CKPT, map_location="cpu"))
export_model.eval().to("cpu")

# 1. Self-contained checkpoint: weights + everything needed to reuse them
checkpoint = {
    "state_dict": export_model.state_dict(),
    "arch": "efficientnet_b0", "num_classes": 1, "img_size": IMG_SIZE,
    "normalize_mean": (0.485, 0.456, 0.406), "normalize_std": (0.229, 0.224, 0.225),
    "output": "single logit; prob_fake = sigmoid(logit)",
    "class_mapping": {0: "REAL", 1: "FAKE"},
    "best_val_auc": float(best_val_auc),
    # str() both versions -- torch.__version__ is a TorchVersion object, not a plain
    # string, and that breaks reloading this dict under weights_only=True later
    "timm_version": str(timm.__version__),
    "torch_version": str(torch.__version__),
}

ckpt_path = os.path.join(CHECKPOINT_DIR, "efficientnet_b0_detector.pt")
torch.save(checkpoint, ckpt_path)
print("Saved portable checkpoint ->", ckpt_path)

# 2. TorchScript: runs without timm / this notebook
example = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
ts_path = os.path.join(CHECKPOINT_DIR, "efficientnet_b0_detector.ts")
with torch.no_grad():
    torch.jit.trace(export_model, example).save(ts_path)
print("Saved TorchScript         ->", ts_path)

# Reload the portable checkpoint and confirm it matches the original
# weights_only=False is safe here since we just wrote this file ourselves
reloaded = timm.create_model("efficientnet_b0", pretrained=False, num_classes=1)
reloaded.load_state_dict(
    torch.load(ckpt_path, map_location="cpu", weights_only=False)["state_dict"])
reloaded.eval()
with torch.no_grad():
    a = torch.sigmoid(export_model(example)).item()
    b = torch.sigmoid(reloaded(example)).item()
assert abs(a - b) < 1e-6, "Reloaded model does not match the original!"
print(f"Reload check passed (example prob_fake={b:.4f}). Export complete.")

# Create the target directory if it doesn't exist
target_replication_dir = "/content/drive/MyDrive/techjam/"
os.makedirs(target_replication_dir, exist_ok=True)

# Copy the exported files to the target directory
shutil.copy(ckpt_path, os.path.join(target_replication_dir, "efficientnet_b0_detector.pt"))
shutil.copy(ts_path, os.path.join(target_replication_dir, "efficientnet_b0_detector.ts"))
print(f"Replicated model files to {target_replication_dir}")

Saved portable checkpoint -> /content/checkpoints/efficientnet_b0_detector.pt
Saved TorchScript         -> /content/checkpoints/efficientnet_b0_detector.ts
Reload check passed (example prob_fake=1.0000). Export complete.
Replicated model files to /content/drive/MyDrive/techjam/
